In [1]:
import pandas as pd
y = pd.read_parquet("../data/processed/hourly_jan2026.parquet")["trips"]

In [ ]:
y

In [3]:
print(type(y.index))

<class 'pandas.DatetimeIndex'>


In [4]:
import sys
sys.path.append("..")
from app.features import make_features

In [5]:
feat = make_features(y)

expected = y.shift(25).rolling(24).mean()
pd.testing.assert_series_equal(expected, feat["ma_24"], check_names=False)
feat.head(3)


,y,hour,dayofweek,is_low_demand,lag_168,ma_24
tpep_pickup_datetime,,,,,,
2026-01-01 00:00:00,9294,0,3,True,NaN,NaN
2026-01-01 01:00:00,10254,1,3,True,NaN,NaN
2026-01-01 02:00:00,7853,2,3,True,NaN,NaN


In [6]:
X = feat.dropna()
y1 = y.loc[X.index]

assert(X.index == y1.index).all()

print("X: ", X.shape, " | y: ", y.shape, " | y1: ", y1.shape)
print("X min index: ", X.index.min())
print(X.head(3)) 

X:  (576, 6)  | y:  (744,)  | y1:  (576,)
X min index:  2026-01-08 00:00:00
                         y  hour  ...  lag_168        ma_24
tpep_pickup_datetime              ...                      
2026-01-08 00:00:00   1876     0  ...   9294.0  4478.541667
2026-01-08 01:00:00    909     1  ...  10254.0  4474.958333
2026-01-08 02:00:00    562     2  ...   7853.0  4474.208333

[3 rows x 6 columns]


In [7]:
import sys
sys.path.append("..")
from app.backtest import walk_forward_splits   

In [8]:
folds = list(walk_forward_splits(X, 24*14, 24, 24, "expanding"))

print("fold: ", len(folds))              
for i, (tr, te) in enumerate(folds[:3]):
    print(f"fold {i}: train {tr.min()} → {tr.max()} | test {te.min()} → {te.max()}")

fold:  10
fold 0: train 2026-01-08 00:00:00 → 2026-01-21 23:00:00 | test 2026-01-22 00:00:00 → 2026-01-22 23:00:00
fold 1: train 2026-01-08 00:00:00 → 2026-01-22 23:00:00 | test 2026-01-23 00:00:00 → 2026-01-23 23:00:00
fold 2: train 2026-01-08 00:00:00 → 2026-01-23 23:00:00 | test 2026-01-24 00:00:00 → 2026-01-24 23:00:00


In [10]:
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

rows = []
for tr, te in folds:
    y_true = y1.loc[te]
    y_pred = X.loc[te, "lag_168"]
    rows.append({"MAE": mean_absolute_error(y_true, y_pred), "MAPE": mean_absolute_percentage_error(y_true, y_pred)})

pd.DataFrame(rows).agg(["mean", "median", "std"])    

,MAE,MAPE
mean,868.833333,0.405253
median,555.291667,0.104801
std,838.182755,0.688735
